# Custom prediction with a trained model

This notebooks shows how to use a fine-tuned model for prediction tasks.

## Imports

In [ ]:
import os
# Set the GPU to the one with available memory (nvidia-smi)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from esnlir.dataset_utils.dataset import BERTDataset
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report

import pandas as pd

## Constants

In [ ]:
# File path of the dataset you want to predict
DATASET = "/data/nli-training-example/data/test.json"

# Folder path of the model you fine-tuned
MODEL = "/data/nli-training-example/models/xlmroberta/model"

In [ ]:
# Maximum length of tokens per sentence
MAX_LEN = 256

# Original model from hugging-faces
MODEL_TYPE = "FacebookAI/xlm-roberta-base"

# Batch size
BATCH_SIZE = 64

# CUDA device
DEVICE = "cuda:0"

# CPU device
CPU_DEVICE = "cpu"

## Execution

### Load the dataset

In [ ]:
dataset = BERTDataset(
    dataframe_file=DATASET,
    max_len=MAX_LEN,
    model_type=MODEL_TYPE,
    only_premise=False,
    max_samples=None
)

In [ ]:
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)

## Load the model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(DEVICE)

## Predict the dataset with the model

In [ ]:
#
model.zero_grad()

# Real labels accumulator
y_true = []
y_pred = []
# Run over all dataset batches
for batch in tqdm(dataloader):
    
    # The model tokenized input
    inputs = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    
    # Get the labels to evaluate
    batch_y_true = batch["labels"].to(CPU_DEVICE).detach().numpy().tolist()
    y_true.extend(batch_y_true)
    
    # Predict using the model
    batch_y_pred = model(inputs, attention_mask=attention_mask).logits.to(CPU_DEVICE).detach().numpy().tolist()
    y_pred.extend(batch_y_pred)
    
    torch.cuda.empty_cache()

In [ ]:
y_true[:2]

In [ ]:
y_pred[:2]

## Generating metrics over predictions

As shown above, the predictions and real values are set as logits, meaning a vector of probability for each class. For each example, the positions are sorted by class name in order.

In [ ]:
dataset.classes

We can use the dataset labels to revert the logits to the original labels and use any standard metric library to evaluate the results

In [ ]:
class_map = {index: label for index, label in enumerate(dataset.classes)}
class_map

In [ ]:
# For each example find the biggest probability position and map it to the class name
y_true_cat = np.vectorize(class_map.get)(np.argmax(y_true, axis=1))
y_true_cat[:2]

In [ ]:
# For each example find the biggest probability position and map it to the class name
y_pred_cat = np.vectorize(class_map.get)(np.argmax(y_pred, axis=1))
y_pred_cat[:2]

## We can use sklearn over these new real and prediction label arrays

### Confusion matrix

In [ ]:
conf_mat = confusion_matrix(y_true_cat, y_pred_cat)

plt.figure()
plt.title("Demo confusion matrix")
ax = plt.gca()
sns.heatmap(conf_mat, annot=True, cmap="mako", ax=ax, xticklabels=dataset.classes, yticklabels=dataset.classes)
plt.show()

## Classification report

In [ ]:
class_report = classification_report(y_true_cat, y_pred_cat, output_dict=True)
df_class_metrics = pd.DataFrame(class_report)
df_class_metrics